In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/clean/prices_clean.csv", parse_dates=["Date"])

In [37]:
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

In [38]:
df.columns

Index(['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')

In [39]:
by_ticker = df.groupby("Ticker")
ret_1 = by_ticker["Close"].pct_change(1)
ret_5 = by_ticker["Close"].pct_change(5)
ret_20 = by_ticker["Close"].pct_change(20)

In [40]:
gap = df["Open"] / by_ticker["Close"].shift(1) - 1
body = df["Close"] / df["Open"] - 1

In [41]:
rng = (df["High"] - df["Low"]).replace(0, np.nan)

In [42]:
close_pos = (df["Close"] - df["Low"]) / rng

In [43]:
ma20 = by_ticker["Close"].transform(lambda s: s.rolling(20).mean())
ma20_ratio = df["Close"] / ma20 - 1

In [44]:
vol_20 = ret_1.groupby(df["Ticker"]).transform(lambda s: s.rolling(20).std())

In [45]:
avg_vol = by_ticker["Volume"].transform(lambda s: s.rolling(20).mean())
vol_ratio = df["Volume"] / avg_vol

In [46]:
target = by_ticker["Close"].shift(-1) / df["Close"] - 1

In [49]:
features = pd.DataFrame({
    "ret_1": ret_1, "ret_5": ret_5, "ret_20": ret_20,
    "gap": gap, "body": body, "close_pos": close_pos,
    "ma20_ratio": ma20_ratio, "vol_20": vol_20,
    "vol_ratio": vol_ratio, "target": target,
})

feat = pd.concat([df[["Date", "Ticker"]], features], axis=1)
feat = feat.replace([np.inf, -np.inf], np.nan).dropna()
feat.to_csv("data/clean/features.csv", index=False)

last record for app

In [52]:
all_rows = pd.concat([df[["Date", "Ticker"]], features], axis=1)
all_rows = all_rows.replace([np.inf, -np.inf], np.nan)

latest = all_rows.groupby("Ticker").tail(1).drop(columns="target")
latest.to_csv("data/clean/latest.csv", index=False)